# 03 — 評価指標・モデル間比較・考察

- 目的はモデルの性能比較。個別の推薦挙動は02で確認する。
- `run_experiment` と共通評価器で評価し、指標・要求ごとの差・実行時間を可視化。
- 既定は合成データ。単独実行可能。MLflowの記録・読込みは無効。

| 入力 | 形式 |
| --- | --- |
| 今回の実行 | `BenchmarkReport` |
| ローカル保存 | `results.json` |
| MLflow成果物 | 集計済み `reports/report.json` |

- 比較条件：同じ `dataset_id`・`dataset_sha256`・`top_k`。
- 現形式のSHA-256は学習履歴・要求・正解を含む。条件不一致・hashなしは拒否。
- データ形式を変更する場合は、分割・候補集合の識別も維持する。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from music_recsys.analysis import compare_reports, request_rows
from music_recsys.evaluation.benchmark import run_experiment, save_report
from music_recsys.evaluation.tracking import TrackingConfig

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)

TRACK_EXPERIMENT = False
tracking = (
    TrackingConfig.from_settings({"enabled": True})
    if TRACK_EXPERIMENT
    else TrackingConfig(enabled=False)
)
report = run_experiment(ROOT / "configs/toy.toml", tracking=tracking)
RUN_DIR = save_report(report, ROOT / "experiments/toy/runs")
requests = request_rows(report)
display(
    {
        "dataset": report.metadata.dataset_id,
        "top_k": report.metadata.top_k,
        "requests": report.summary.n_requests,
        "evaluated": report.summary.n_evaluated,
        "excluded": report.summary.n_excluded,
        "failed": report.summary.n_failed,
        "short_outputs": report.summary.n_short,
    }
)
print("Saved report:", RUN_DIR / "results.json")

## 要求別の指標と集計対象

| 要求 | NDCG・Recallの扱い |
| --- | --- |
| 候補内に正解なし | 未定義（`None`）。平均から除外 |
| 正解あり・推薦失敗 | 0。平均・失敗数に含める |

- 合成データの要求5は正解なし。
- Cold userを含む動作確認。Music4AllのWarm対象評価とは異なる。

In [ ]:
display(requests)
evaluated = [row for row in requests if row["ndcg"] is not None]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
indices = [row["request_index"] for row in evaluated]
axes[0].bar(
    [i - 0.18 for i in indices],
    [row["ndcg"] for row in evaluated],
    width=0.36,
    color="#256b78",
    label="NDCG",
)
axes[0].bar(
    [i + 0.18 for i in indices],
    [row["recall"] for row in evaluated],
    width=0.36,
    color="#c77936",
    label="Recall",
)
axes[0].set(
    title=f"Evaluated requests, K={report.metadata.top_k}",
    xlabel="Request index",
    ylabel="Score",
    ylim=(0, 1.05),
    xticks=indices,
)
axes[0].legend()
axes[1].bar(
    ["Evaluated", "Excluded", "Failed"],
    [report.summary.n_evaluated, report.summary.n_excluded, report.summary.n_failed],
    color=["#256b78", "#c77936", "#8b507b"],
)
axes[1].set(title="Evaluation population and failures", ylabel="Requests")
fig.tight_layout()
plt.show()

## 任意：保存済みreport・MLflow成果物の追加

| 設定 | 用途 |
| --- | --- |
| `ADDITIONAL_REPORTS` | 表示名とローカルreportのPath |
| `READ_MLFLOW=True` / `MLFLOW_RUN_ID` | 起動済みMLflowから集計成果物を取得 |

- 既定は実装済みのPopularity 1モデル。
- 同条件で保存した別モデルの結果を追加可能。
- ユーザー別結果・生履歴はMLflowへ送信しない。
- 接続設定：[MLflowの手順](../docs/mlflow.md)。

In [ ]:
ADDITIONAL_REPORTS = {}  # 例: {"saved-popularity": Path("/absolute/path/results.json")}
READ_MLFLOW = False
MLFLOW_RUN_ID = None

reports = {"Popularity (current)": report, **ADDITIONAL_REPORTS}
if READ_MLFLOW:
    from mlflow import MlflowClient

    if MLFLOW_RUN_ID is None:
        raise ValueError("Set MLFLOW_RUN_ID before reading an MLflow report")
    client = MlflowClient(tracking_uri=TrackingConfig.from_settings().tracking_uri)
    mlflow_dir = RUN_DIR / "mlflow-download"
    mlflow_dir.mkdir(exist_ok=True)
    artifact_path = client.download_artifacts(MLFLOW_RUN_ID, "reports/report.json", str(mlflow_dir))
    reports["MLflow saved run"] = Path(artifact_path)

comparison = compare_reports(reports)
display(comparison)

## 共通条件での指標・実行時間

- 比較表にモデル・seed・学習時間を表示。条件不一致は作図前にエラー。
- 時間は実測値。短い合成実験だけで速度の優劣は判断しない。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
labels = [row["label"] for row in comparison]
for ax, metric, title in zip(
    axes,
    ("ndcg", "recall", "mean_recommendation_seconds"),
    (
        f"NDCG@{report.metadata.top_k}",
        f"Recall@{report.metadata.top_k}",
        "Mean recommendation seconds",
    ),
):
    values = [row[metric] for row in comparison]
    defined = [(label, value) for label, value in zip(labels, values) if value is not None]
    ax.bar([label for label, _ in defined], [value for _, value in defined], color="#256b78")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=15)
    if metric in ("ndcg", "recall"):
        ax.set_ylim(0, 1.05)
    if not defined:
        ax.text(0.5, 0.5, "Undefined: no evaluated requests", ha="center", transform=ax.transAxes)
fig.tight_layout()
plt.show()

## 考察・次の比較

- 合成データのPopularity：NDCG@2 = 0.561019、Recall@2 = 0.625（5要求中4要求）。
- Music4Allでの性能比較は未実施。分割・最小履歴量・最終再学習の条件は未決定。
- 別モデルは同じデータ・分割・Kで比較する。
- LLM／Agentの費用・説明品質は未測定。